In [1]:
import torch
from torchvision import datasets,transforms
from torch.utils.data import DataLoader,random_split

In [3]:
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
data_dir = "/kaggle/input/datasets/karakaggle/kaggle-cat-vs-dog-dataset/kagglecatsanddogs_3367a/PetImages"
full_dataset = datasets.ImageFolder(root=data_dir, transform = transform)

train_size = int(0.8*len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
testloader = DataLoader(test_dataset, batch_size=32)

In [6]:
print(f"total images: {len(full_dataset)}")
print(f"train images: {len(train_dataset)}")
print(f"test images: {len(test_dataset)}")

total images: 24959
train images: 19967
test images: 4992


In [7]:
import torch.nn as nn
import torch.optim as optim

In [17]:
#Build the CNN

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(128*16*16,512),
            nn.ReLU(),
            nn.Linear(512,2)
        )
    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(-1, 128*16*16)
        x = self.fc_layers(x)
        return x

In [18]:
device = torch.device("cuda")
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [21]:
#Training the CNN
epochs = 10
for epoch in range(epochs):
    epoch_training_loss = 0.0
    for images,labels in trainloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model.forward(images)
        loss = criterion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss += loss.item()
    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 0.3739325402495571
epoch = 2/10 & loss = 0.27888204974050707
epoch = 3/10 & loss = 0.17209205746602935
epoch = 4/10 & loss = 0.08697114212144441
epoch = 5/10 & loss = 0.04580307617214156
epoch = 6/10 & loss = 0.02878801396983386
epoch = 7/10 & loss = 0.031655707020974706
epoch = 8/10 & loss = 0.025087178034477978
epoch = 9/10 & loss = 0.030106092236903165
epoch = 10/10 & loss = 0.0259455833998415


In [22]:
#evaluate our CNN

correct_labels = 0
total_labels = 0
model.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)
        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)
print(f"accuracy={correct_labels/total_labels*100}")
    

accuracy=83.45352564102564
